# tensor-zeros-init — ex7: confusion matrix from (pred, true) pairs

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-zeros-init`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch zero-init — quick refresher

**The allocate-then-scatter pattern.** Pre-allocate a buffer of the right `(shape, dtype, device)` with `t.zeros(...)`, then write per-element results into it via indexed assignment or `index_add_` / `scatter_add_`. This is faster and clearer than `list.append` + `t.stack`, and it's the canonical move for histograms, confusion matrices, depth buffers, and any per-ray accumulator.

**Dtype matters.** Default is `float32`. Index buffers MUST be `t.long`. Counters should be `t.long` (or `t.int64`). Use `t.zeros_like(x)` when you want a fresh buffer that mirrors `x.shape + x.dtype + x.device` exactly.

### Exercise 7 — confusion matrix from (pred, true) pairs

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Allocate a `(C, C)` zero buffer and accumulate `(pred, true)` pairs into it via 2-D scatter; visualize the matrix as a heatmap.
> Keywords: confusion-matrix, scatter-add, visualization, classification
> ```

**KCs targeted:** `zeros-multi-axis-shape`, `zeros-dtype-control`, `zeros-allocate-then-fill`

Implement `ex7_confusion_matrix(preds, trues, n_classes)`. The canonical classification-debug pattern:

1. Allocate a `(n_classes, n_classes)` zero matrix with `dtype=t.long`. Convention: row = predicted class, column = true class.
2. For each `(p, y)` pair, increment `cm[p, y]` by 1.
3. Return the matrix.

Trick: a 2-D scatter is most cleanly done by flattening. Compute `flat_idx = preds * n_classes + trues`, allocate a flat `(n_classes * n_classes,)` buffer, scatter-add 1 per index, then `.view(n_classes, n_classes)`. This trains the allocate-then-reshape idiom you'll use again for occupancy grids and voxel volumes.

Inputs:
- `preds`: 1-D `t.long`, values in `[0, n_classes)`.
- `trues`: 1-D `t.long`, same shape as `preds`.
- `n_classes`: int.

Output: `(n_classes, n_classes)` `t.long` tensor. Diagonal entries are the correct-prediction counts.

The visualization below the solution renders the matrix as a matplotlib heatmap so you can read off the misclassification patterns visually.

In [ ]:
def ex7_confusion_matrix(preds: Tensor, trues: Tensor, n_classes: int) -> Tensor:
    """Allocate (C, C) long zeros; scatter (pred, true) counts; return matrix."""
    raise NotImplementedError()


def _test_ex7():
    # 3 classes, 8 predictions. 5 correct (diag), 3 wrong.
    preds = t.tensor([0, 1, 2, 0, 1, 2, 1, 2], dtype=t.long)
    trues = t.tensor([0, 1, 2, 0, 0, 1, 2, 0], dtype=t.long)
    cm = ex7_confusion_matrix(preds, trues, n_classes=3)
    assert cm.shape == (3, 3), f'expected (3,3), got {tuple(cm.shape)}'
    assert cm.dtype == t.long, f'expected dtype long, got {cm.dtype}'
    # Manual count: rows=pred, cols=true.
    #   pred=0,true=0: 2  pred=1,true=1: 1  pred=2,true=2: 1
    #   pred=1,true=0: 1  pred=1,true=2: 1  pred=2,true=0: 1  pred=2,true=1: 1
    expected = t.tensor([
        [2, 0, 0],
        [1, 1, 1],
        [1, 1, 1],
    ], dtype=t.long)
    assert t.equal(cm, expected), f'value mismatch:\n{cm}\nvs\n{expected}'
    assert cm.sum().item() == len(preds), 'matrix total must equal n_samples'
    assert cm.diag().sum().item() == 4, f'expected 4 correct on diagonal, got {cm.diag().sum().item()}'

    # --- Heatmap visualization ---
    rng = t.Generator().manual_seed(7)
    n_classes = 5
    big_trues = t.randint(0, n_classes, (300,), generator=rng)
    # Simulate a noisy classifier: 70% correct, 30% random.
    noise_mask = t.rand(300, generator=rng) < 0.3
    big_preds = t.where(noise_mask, t.randint(0, n_classes, (300,), generator=rng), big_trues)
    big_cm = ex7_confusion_matrix(big_preds, big_trues, n_classes=n_classes)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(big_cm.numpy(), cmap='Blues')
    ax.set_xlabel('true class')
    ax.set_ylabel('predicted class')
    ax.set_title(f'ex7 confusion matrix (300 samples, ~70% acc)')
    ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, str(big_cm[i, j].item()), ha='center', va='center',
                    color='white' if big_cm[i, j].item() > big_cm.max().item() / 2 else 'black')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex7')
    print("ex7 ✓")

_test_ex7()

<details><summary>Solution</summary>

```python
def ex7_confusion_matrix(preds: Tensor, trues: Tensor, n_classes: int) -> Tensor:
    flat_idx = preds * n_classes + trues
    flat = t.zeros(n_classes * n_classes, dtype=t.long)
    flat.index_add_(0, flat_idx, t.ones_like(flat_idx))
    return flat.view(n_classes, n_classes)
```

**The 2-D-via-flat trick.** PyTorch's `index_add_` only takes 1-D indices into a 1-D output, so a 2-D scatter is done by linearising `(row, col) → row * n_cols + col`, scattering into a flat buffer, then reshaping back. This is the SAME pattern used to splat fragments into a 2-D framebuffer in Ray Tracing.

**Why diagonal sum = accuracy * N.** Every `(pred==true)` pair lands on the diagonal. `cm.diag().sum() / cm.sum()` is the accuracy.

**The integrative load.** Three KCs at once: multi-axis allocation, long-dtype counter, and indexed scatter — Lohr et al. ITiCSE 2025 shows 3-KC exercises drop to ~40% solvability. Expect to look things up.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()